In [ ]:
from cyvcf2 import VCF

vcf_path = "/mnt/d/thesis/data/clinvar_selected_variants.vcf"
vcf_reader = VCF(vcf_path)

# samples for inspection
for i, variant in enumerate(vcf_reader):
    print(f"Variant {i+1}:")
    print("  CHROM:", variant.CHROM)
    print("  POS:", variant.POS)
    print("  REF:", variant.REF)
    print("  ALT:", variant.ALT)
    print("  INFO:", variant.INFO)
    print("-" * 50)
    if i >= 4:
        break


In [ ]:
vcf_reader = VCF(vcf_path)  # reload each time

for i, variant in enumerate(vcf_reader):
    info_dict = dict(variant.INFO)  # transform INFO into dict
    print(f"Variant {i+1} INFO fields:")
    for key, value in info_dict.items():
        print(f"  {key}: {value}")
    print("-" * 50)
    if i >= 4:
        break


In [ ]:
# inspect unique values in CLNSIG and CLNDN to determain filtering strategy for next step
import pandas as pd
vcf_reader = VCF(vcf_path)

# initialize sets
clnsig_values = set()
clndn_values = set()

# unique values
for i, variant in enumerate(vcf_reader):
    clnsig = variant.INFO.get("CLNSIG")
    clndn = variant.INFO.get("CLNDN")
    
    if clnsig:
        clnsig_values.add(clnsig)
    if clndn:
        clndn_values.add(clndn)

list(sorted(clnsig_values)), list(sorted(clndn_values))



In [ ]:
# save to csv for further inspection
df_clndn = pd.DataFrame(sorted(clndn_values), columns=["CLNDN"])
df_clndn.to_csv("unique_clndn_values.csv", index=False)

In [ ]:
import re

vcf_reader = VCF(vcf_path)
filtered_variants = []

for variant in vcf_reader:
    clnsig = variant.INFO.get("CLNSIG", "")
    clndn = variant.INFO.get("CLNDN", "")
    
    # filter with key words: benign/pathogenic, and breast cancer
    if (
        isinstance(clnsig, str) and isinstance(clndn, str) and
        re.search(r"benign|pathogenic", clnsig, re.IGNORECASE) and
        re.search(r"breast", clndn, re.IGNORECASE)
    ):
        filtered_variants.append({
            "CHROM": variant.CHROM,
            "POS": variant.POS,
            "REF": variant.REF,
            "ALT": ",".join(variant.ALT),
            "CLNSIG": clnsig,
            "CLNDN": clndn
        })

print(f"Total filtered variants: {len(filtered_variants)}")

for v in filtered_variants[:5]:
    print(v)


In [ ]:
from cyvcf2 import VCF

vcf_path = "../data/clinvar_selected_variants.vcf"

vcf_reader = VCF(vcf_path)

count = 0
for v in vcf_reader:
    clnsig = v.INFO.get("CLNSIG", "")
    clndn = v.INFO.get("CLNDN", "")
    if "breast" in clndn.lower():
        print("CLNDN:", clndn)
        print("CLNSIG:", clnsig)
        print("-" * 40)
        count += 1
        if count >= 10:
            break


## Second try

In [9]:
from cyvcf2 import VCF
from collections import Counter

# load vcf
vcf_path = "../data/clinvar_selected_variants.vcf"
vcf_reader = VCF(vcf_path)

# initiate counter
clnsig_counter = Counter()

# find unique value and counte
for variant in vcf_reader:
    clnsig = variant.INFO.get("CLNSIG")
    if clnsig:
        clnsig_str = str(clnsig).strip()
        clnsig_counter[clnsig_str] +=1

vcf_reader.close()

# print
print("Unique CLNSIG values and their counts:")
for clnsig_value, count in clnsig_counter.most_common():
    print(f"{clnsig_value}: {count}")

Unique CLNSIG values and their counts:
Conflicting_classifications_of_pathogenicity: 7675
Likely_benign: 6109
Pathogenic: 2019
Benign/Likely_benign: 1653
Benign: 1556
Pathogenic/Likely_pathogenic: 667
Likely_pathogenic: 420
Uncertain_significance: 1
not_provided: 1
Conflicting_classifications_of_pathogenicity|risk_factor: 1


In [10]:
# further filter
from cyvcf2 import VCF, Writer

## define vcf files paths
vcf_path = "../data/clinvar_selected_variants.vcf"
output_vcf_path = "../data/clinvar_bc_variants_for_vep.vcf"

## define labled allowed in CLNSIG
allowed_labels = [
    "Likely_benign",
    "Pathogenic",
    "Benign/Likely_benign",
    "Benign",
    "Pathogenic/Likely_pathogenic",
    "Likely_pathogenic"
]

## initiate VCF reader and writer
vcf_reader = VCF(vcf_path)
vcf_writer = Writer(output_vcf_path, vcf_reader)

## filtering
for variant in vcf_reader:
    clnsig = variant.INFO.get("CLNSIG")
    if clnsig is not None:
        clnsig_str = str(clnsig).strip() # clean string
        if clnsig_str in allowed_labels:
            vcf_writer.write_record(variant)

## close
vcf_reader.close()
vcf_writer.close()

## note
print("Filtering has been done, all qualified variants are in: ", output_vcf_path)

Filtering has been done, all qualified variants are in:  ../data/clinvar_bc_variants_for_vep.vcf


In [11]:
from cyvcf2 import VCF
from collections import Counter

# load vcf
vcf_path = "../data/clinvar_bc_variants_for_vep.vcf"
vcf_reader = VCF(vcf_path)

# initiate counter
clnsig_counter = Counter()

# find unique value and counte
for variant in vcf_reader:
    clnsig = variant.INFO.get("CLNSIG")
    if clnsig:
        clnsig_str = str(clnsig).strip()
        clnsig_counter[clnsig_str] +=1

vcf_reader.close()

# print
print("Unique CLNSIG values and their counts:")
for clnsig_value, count in clnsig_counter.most_common():
    print(f"{clnsig_value}: {count}")

Unique CLNSIG values and their counts:
Likely_benign: 6109
Pathogenic: 2019
Benign/Likely_benign: 1653
Benign: 1556
Pathogenic/Likely_pathogenic: 667
Likely_pathogenic: 420
